In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection    import train_test_split, KFold, cross_val_score
from sklearn.compose            import ColumnTransformer
from sklearn.pipeline           import Pipeline
from sklearn.impute             import SimpleImputer
from sklearn.preprocessing      import OneHotEncoder, StandardScaler
from sklearn.linear_model       import Ridge, Lasso, ElasticNet, LinearRegression, RidgeCV, LassoCV
from sklearn.metrics            import mean_absolute_percentage_error
from xgboost                    import XGBRegressor

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Imports OK')

Imports OK


In [2]:
df = pd.read_csv('data/house_dataset.csv')
print('Shape:', df.shape)
df.head(3)

Shape: (9200, 18)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.5,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.5,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.0,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA


## preprocessing

In [3]:
def clean(df):
    df = df.copy()

    # Parse date
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['sale_year']  = df['date'].dt.year
    df['sale_month'] = df['date'].dt.month
    df = df.drop(columns=['date'])

    # DROP zero-price rows — these are data errors, MAPE divides by y_true
    before = len(df)
    df = df[df['price'] > 0]
    print(f'Dropped {before - len(df)} zero-price rows. Remaining: {len(df)}')

    # yr_renovated: 0 means never renovated
    df['yr_renovated'] = df['yr_renovated'].replace(0, np.nan)

    # Clip bedroom outlier
    df['bedrooms'] = df['bedrooms'].clip(upper=10)

    # Drop zero-signal columns
    df = df.drop(columns=['street', 'country'], errors='ignore')

    return df

df = clean(df)
print('Shape after cleaning:', df.shape)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head(2)

Dropped 98 zero-price rows. Remaining: 9102
Shape after cleaning: (9102, 17)

Missing values:
yr_renovated    5412
dtype: int64


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,city,statezip,sale_year,sale_month
0,313000.0,3.0,1.5,1340,7912,1.5,0,0,3,1340,0,1955,2005.0,Shoreline,WA 98133,2014,5
1,2384000.0,5.0,2.5,3650,9050,2.0,0,4,5,3370,280,1921,NaN,Seattle,WA 98119,2014,5


## train/test split 80/20

In [4]:
X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# Log-transform target — reduces MAPE and stabilises variance
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

print('Train:', X_train.shape)
print('Test: ', X_test.shape)
print('y_train range: ${:,.0f} — ${:,.0f}'.format(y_train.min(), y_train.max()))
print('y_train_log range: {:.2f} — {:.2f}'.format(y_train_log.min(), y_train_log.max()))

Train: (7281, 16)
Test:  (1821, 16)
y_train range: $7,800 — $26,590,000
y_train_log range: 8.96 — 17.10


### basic feature engineering

In [5]:
# ── Column definitions ────────────────────────────────────────
numeric_features = [
    'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
    'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
    'yr_built', 'yr_renovated',   # yr_renovated NaNs handled by imputer
    'sale_year', 'sale_month'
]

categorical_features = [
    'city',      # ~70 unique values — OHE is fine
    'statezip',  # ~70 unique values — OHE is fine
    # 'street' and 'country' already dropped in clean()
]

# ── Sub-pipelines ─────────────────────────────────────────────
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # handles yr_renovated NaNs
    ('scaler',  StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop')  # drop anything not listed above

print('Preprocessor defined.')
print(f'  Numeric features    : {len(numeric_features)}')
print(f'  Categorical features: {len(categorical_features)}')

Preprocessor defined.
  Numeric features    : 14
  Categorical features: 2


In [6]:
# Fit preprocessor on train, transform train
preprocessor.fit(X_train)
X_train_transformed = preprocessor.transform(X_train)

print('Transformed shape:', X_train_transformed.shape)
print('Any NaN in transformed?', np.isnan(X_train_transformed).any())
print('Any Inf in transformed?', np.isinf(X_train_transformed).any())
print('Value range: {:.2f} to {:.2f}'.format(
    X_train_transformed.min(), X_train_transformed.max()
))
# Expected: no NaN, no Inf, values roughly in range -5 to 5 (standardised)

Transformed shape: (7281, 135)
Any NaN in transformed? False
Any Inf in transformed? False
Value range: -6.08 to 31.66


In [7]:
models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000)
}

## train

In [8]:
results_log = []

y_train_log = np.log1p(y_train)

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train_log)
    
    log_preds = pipe.predict(X_test)
    preds = np.expm1(log_preds)   # convert back to normal price scale
    preds = np.maximum(preds, 1)
    
    mape = mean_absolute_percentage_error(y_test, preds)
    
    results_log.append({
        "model": name,
        "target_type": "log_price",
        "MAPE": mape
    })

results_log_df = pd.DataFrame(results_log).sort_values("MAPE")
results_log_df

,model,target_type,MAPE
1,Ridge,log_price,0.169448
0,OLS,log_price,0.169499
3,ElasticNet,log_price,0.173399
2,Lasso,log_price,0.178843


In [9]:
all_results = results_log_df.sort_values("MAPE")
best_row = all_results.iloc[0]
print("Best model:")
print(best_row)

Best model:
model              Ridge
target_type    log_price
MAPE            0.169448
Name: 1, dtype: object


## xgboost baseline

In [11]:
from xgboost import XGBRegressor
from sklearn.preprocessing import OrdinalEncoder

# XGBoost-specific preprocessor — no scaling, ordinal encode cats
xgb_preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features),
], remainder='drop')

xgb_pipe = Pipeline([
    ('preprocessor', xgb_preprocessor),
    ('model', XGBRegressor(
        n_estimators=500,       # more trees — use early stopping to avoid overfitting
        max_depth=5,            # slightly shallower than default to reduce overfitting
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0,
    )),
])

# Train on log price — same target as linear models
xgb_pipe.fit(X_train, y_train_log)

# Convert back to price space before MAPE
xgb_preds = np.expm1(xgb_pipe.predict(X_test))
xgb_preds = np.maximum(xgb_preds, 1)

xgb_mape = mean_absolute_percentage_error(y_test, xgb_preds)
print(f"XGBoost MAPE: {xgb_mape*100:.2f}%")
print(f"Linear (Ridge) MAPE: {16.94:.2f}%")  # replace with your actual variable
print(f"Gap: {xgb_mape*100 - 16.94:.2f}pp")

XGBoost MAPE: 12.54%
Linear (Ridge) MAPE: 16.94%
Gap: -4.40pp
